### LIBRARY IMPORTS

In [1]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
import copy

from sklearn.metrics import accuracy_score

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.cnn_regressor import CNNRegressor

### CONFIGURATION

In [2]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = modeling_config["main"]["active_dataset"]
active_dataset_config = datasets_config[active_dataset]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, valid, test = data_manager.load_image_data(
    active_dataset
)

y_test = np.asarray(test.dataset.targets)[test.indices]

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)

y_train, y_valid = processor.transform_target(y_train, y_valid)

test = processor.convert_to_numpy(test) 

### GRADIENT BOOSTING

In [7]:
gb_preds = pd.read_csv(f"{MODELS_PATH}/{active_dataset}/2026_03_27_08_55/predictions.csv")
accuracy_score(y_test, gb_preds[active_dataset_config["target"]])

0.9903

### CONVOLUTIONAL NEURAL NETWORKS

Big CNN total parameters: 46.730

Small CNN total parameters: 3.154

In [13]:
class CNN(CNNRegressor):
    def __init__(self, **hyperparameters):
        super().__init__(**hyperparameters)

    def fit(self, X_train, y_train, X_valid, y_valid, patience=5):
        
        in_channels = X_train.shape[1]
        output_size = int(np.max(y_train)) + 1

        image_size = X_train.shape[-1]

        conv1_out = image_size - self.kernel_size + 1
        pool1_out = conv1_out // self.pool_size

        conv2_out = pool1_out - self.kernel_size + 1
        pool2_out = conv2_out // self.pool_size

        linear_input = self.channels[1] * pool2_out * pool2_out
        
        self._get_network(in_channels, linear_input, output_size)

        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).long().to(self.device).view(-1)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), self.learning_rate)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                
                loss = criterion(preds, batch_y.long().view(-1))
                loss.backward()
                optimizer.step()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

                pred_labels = val_preds.argmax(dim=1)
                val_acc = (pred_labels == y_valid_t).float().mean().item()

            if (epoch + 1) % 1 == 0:
                print(f"Epoch: {epoch + 1} | Validation Log Loss: {val_loss:.4f} | Validation Accuracy: {val_acc:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)

In [15]:
big_cnn = CNN(epochs=100, learning_rate=0.001, channels=[16, 32], kernel_size=5, pool_size=2, hidden_size=64, batch_size=256)
big_cnn.fit(X_train, y_train, X_valid, y_valid)

print ("-" * 50)

raw_preds = big_cnn.predict(test) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)

score = accuracy_score(y_test, preds)
print(f"Test Accuracy: {score:.4f}")

Epoch: 1 | Validation Log Loss: 0.1769 | Validation Accuracy: 0.9499
Epoch: 2 | Validation Log Loss: 0.1038 | Validation Accuracy: 0.9703
Epoch: 3 | Validation Log Loss: 0.0722 | Validation Accuracy: 0.9791
Epoch: 4 | Validation Log Loss: 0.0649 | Validation Accuracy: 0.9810
Epoch: 5 | Validation Log Loss: 0.0651 | Validation Accuracy: 0.9800
Epoch: 6 | Validation Log Loss: 0.0500 | Validation Accuracy: 0.9852
Epoch: 7 | Validation Log Loss: 0.0493 | Validation Accuracy: 0.9852
Epoch: 8 | Validation Log Loss: 0.0437 | Validation Accuracy: 0.9867
Epoch: 9 | Validation Log Loss: 0.0460 | Validation Accuracy: 0.9857
Epoch: 10 | Validation Log Loss: 0.0528 | Validation Accuracy: 0.9845
Epoch: 11 | Validation Log Loss: 0.0412 | Validation Accuracy: 0.9888
Epoch: 12 | Validation Log Loss: 0.0386 | Validation Accuracy: 0.9887
Epoch: 13 | Validation Log Loss: 0.0472 | Validation Accuracy: 0.9857
Epoch: 14 | Validation Log Loss: 0.0419 | Validation Accuracy: 0.9883
Epoch: 15 | Validation Log Lo

In [16]:
small_cnn = CNN(epochs=100, learning_rate=0.001, channels=[4, 8], kernel_size=5, pool_size=2, hidden_size=16, batch_size=256)
small_cnn.fit(X_train, y_train, X_valid, y_valid)

print ("-" * 50)

raw_preds = small_cnn.predict(test) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)

score = accuracy_score(y_test, preds)
print(f"Test Accuracy: {score:.4f}")

Epoch: 1 | Validation Log Loss: 0.3901 | Validation Accuracy: 0.8843
Epoch: 2 | Validation Log Loss: 0.2523 | Validation Accuracy: 0.9272
Epoch: 3 | Validation Log Loss: 0.1962 | Validation Accuracy: 0.9433
Epoch: 4 | Validation Log Loss: 0.1624 | Validation Accuracy: 0.9530
Epoch: 5 | Validation Log Loss: 0.1430 | Validation Accuracy: 0.9580
Epoch: 6 | Validation Log Loss: 0.1356 | Validation Accuracy: 0.9594
Epoch: 7 | Validation Log Loss: 0.1225 | Validation Accuracy: 0.9631
Epoch: 8 | Validation Log Loss: 0.1144 | Validation Accuracy: 0.9668
Epoch: 9 | Validation Log Loss: 0.1001 | Validation Accuracy: 0.9708
Epoch: 10 | Validation Log Loss: 0.0994 | Validation Accuracy: 0.9703
Epoch: 11 | Validation Log Loss: 0.0987 | Validation Accuracy: 0.9688
Epoch: 12 | Validation Log Loss: 0.0905 | Validation Accuracy: 0.9737
Epoch: 13 | Validation Log Loss: 0.0863 | Validation Accuracy: 0.9750
Epoch: 14 | Validation Log Loss: 0.0803 | Validation Accuracy: 0.9759
Epoch: 15 | Validation Log Lo